In [ ]:
import random
import re
from datetime import datetime
from zoneinfo import ZoneInfo
import difflib

In [ ]:
BOT_NAME = "AAIT Helpdesk Bot"

EXIT_COMMANDS = {"exit", "quit", "bye", "stop"}

UNKNOWN_RESPONSES = [
    "Sorry, I didn't understand that.",
    "Please ask about tools, lab, attendance, or submission."
]

In [ ]:
TOOL_DETAILS = {
    "trello": "Trello is a project management tool used for planning tasks using boards and cards.",
    "figma": "Figma is a UI/UX design tool used to create wireframes and interface designs.",
    "colab": "Google Colab is an online Python coding platform mainly used for machine learning.",
    "visual paradigm": "Visual Paradigm is used to design UML diagrams and software architecture.",
    "vp": "Visual Paradigm is used to design UML diagrams and software architecture.",
    "tableau": "Tableau is a data visualization tool used to create dashboards and charts."
}

In [ ]:
INTENTS = {

    "greeting": {
        "patterns": ["hi", "hello", "hey", "good morning"],
        "responses": [
            "Hello! I am the AAIT Helpdesk Bot.",
            "Hi! Ask me about tools, lab, attendance or submissions."
        ]
    },

    "tools_info": {
        "patterns": ["trello", "figma", "colab", "vp", "visual paradigm", "tableau"],
        "responses": [
            "AAIT tools include Trello, Figma, Colab, Visual Paradigm, and Tableau."
        ]
    },

    "attendance_info": {
        "patterns": ["attendance", "percentage", "minimum attendance"],
        "responses": [
            "Minimum 75% attendance is required."
        ]
    },

    "lab_location": {
        "patterns": ["lab location", "where is lab", "lab room"],
        "responses": [
            "AAIT Lab is located in Block A, 2nd Floor."
        ]
    },

    "experiment_list": {
        "patterns": ["experiments", "experiment list", "lab experiments"],
        "responses": [
            "Experiments include chatbot, ML tools, and visualization."
        ]
    },

    "submission": {
        "patterns": ["submit", "submission", "upload", "google form"],
        "responses": [
            "Submission is through Google Form or link upload."
        ]
    },

    "time_now": {
        "patterns": ["time", "current time"],
        "responses": []
    },

    "help": {
        "patterns": ["help", "what can you do"],
        "responses": [
            "I can explain tools, lab, attendance, submissions and show current time."
        ]
    }
}

In [ ]:
def preprocess(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def match_intent(user_text, context):
    text = preprocess(user_text)
    tokens = set(text.split())

    best_intent = None
    best_score = 0

    for intent, data in INTENTS.items():
        score = 0
        patterns = data["patterns"]

        # Exact matching
        for pat in patterns:
            pat_clean = preprocess(pat)

            if " " in pat_clean:
                if pat_clean in text:
                    score += 2
            else:
                if pat_clean in tokens:
                    score += 1

        # Fuzzy matching (only if no exact match)
        if score == 0:
            for word in tokens:
                close = difflib.get_close_matches(word, patterns, n=1, cutoff=0.85)
                if close:
                    score += 1
                    break

        if score > best_score:
            best_score = score
            best_intent = intent

    # Memory follow-up feature
    if best_score == 0 and context.get("last_intent") == "tools_info":
        return "tools_info"

    return best_intent if best_score > 0 else "unknown"

In [ ]:
def get_response(user_text, context):
    intent = match_intent(user_text, context)
    context["last_intent"] = intent
    text = preprocess(user_text)

    # Accurate IST Time
    if intent == "time_now":
        now = datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%I:%M:%S %p")
        return f"Current time is {now} IST"

    # Tool-specific explanation
    if intent == "tools_info":
        for tool in TOOL_DETAILS:
            if tool in text:
                return TOOL_DETAILS[tool]
        return "AAIT tools include Trello, Figma, Colab, Visual Paradigm, and Tableau."

    if intent == "unknown":
        return random.choice(UNKNOWN_RESPONSES)

    return random.choice(INTENTS[intent]["responses"])

In [ ]:
def chat():
    print(f"{BOT_NAME}: Hi! Type 'exit' to quit.")
    context = {"last_intent": None}

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            print("Please type something.")
            continue

        cleaned = preprocess(user_input)

        if cleaned in EXIT_COMMANDS:
            print(f"{BOT_NAME}: Goodbye!")
            break

        reply = get_response(user_input, context)
        print(f"{BOT_NAME}: {reply}")

In [ ]:
chat()

AAIT Helpdesk Bot: Hi! Type 'exit' to quit.
You: hello
AAIT Helpdesk Bot: Hi! Ask me about tools, lab, attendance or submissions.
You: what is trello
AAIT Helpdesk Bot: Trello is a project management tool used for planning tasks using boards and cards.
You: attendance
AAIT Helpdesk Bot: Minimum 75% attendance is required.
You: time
AAIT Helpdesk Bot: Current time is 11:54:17 AM IST
You: where is lab
AAIT Helpdesk Bot: AAIT Lab is located in Block A, 2nd Floor.
You: exit
AAIT Helpdesk Bot: Goodbye!
